# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

> **Method:** Logistic Regression as the readable first model, then Random Forest as the stronger comparison.
>
> **Why:** Per the `training-honest-models` skill, my question is a "yes/no with an observed label" shape — Lane 4 (Week 2) already framed this as binary classification producing a probability score for ranking. The skill recommends starting readable (Logistic Regression) and adding complexity only if it earns its keep (Random Forest). Both models are evaluated with **Precision@20/@50**, the same ranking metric my Week-4 baseline used, so the comparison is apples-to-apples.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

> **Split:** `GroupShuffleSplit` on `client_id`, 75/25. Grouped, not random-row, so no page from a held-out client is ever seen in training — the same client-holdout logic notebook 02 and Week 3's contract established. A random row split would let the model implicitly memorize client-specific baseline CTR levels, which is a subtler form of leakage than the label-derived-column kind.
>
> **Label:** Reusing the Lane 4 proxy defined in Week 2 — a page is a genuine CTR anomaly (`is_ctr_anomaly = 1`) if its CTR sits meaningfully below the median CTR of its peer group (same `position_tier` + `content_type`), at sufficient impression volume to avoid low-traffic noise. This is the same target concept my Week-4 rule was trying to catch — so scoring the rule against it on held-out data is a fair test, not a different question in disguise.

In [1]:
import pandas as pd
import os
from sklearn.model_selection import GroupShuffleSplit

# --- Load data (same path logic as previous weeks) ---
file_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(file_path):
    file_path = "https://raw.githubusercontent.com/Bibek-Dhakal/applied-search-intelligence/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)

# Same visibility filter used since Week 1
df = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

# --- Label: CTR anomaly vs peer group (position_tier + content_type) ---
peer_median = df.groupby(["position_tier", "content_type"])["ctr"].transform("median")
df["is_ctr_anomaly"] = (df["ctr"] < 0.5 * peer_median).astype(int)

print(f"Rows after filtering: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Label rate (is_ctr_anomaly = 1): {df['is_ctr_anomaly'].mean():.3f}")

# --- Client-holdout split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"\nTrain: {len(train_df):,} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} rows, {test_df['client_id'].nunique()} clients")

assert set(train_df["client_id"]).isdisjoint(set(test_df["client_id"])), "Client leak between splits!"
print("No client appears in both splits ✓")

Rows after filtering: 22,006
Unique clients: 30
Label rate (is_ctr_anomaly = 1): 0.316

Train: 17,396 rows, 22 clients
Test:  4,610 rows, 8 clients
No client appears in both splits ✓


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

> **Features:** reusing the exact 5-feature set from my Week-3 data contract — `impressions_90d`, `avg_position`, `content_age_days`, `word_count`, `engagement_rate` — all knowable at the decision moment. I deliberately exclude `ctr` and `clicks_90d` here: `ctr` is literally the numerator of my label, so including it would be the same leakage trap from Week 2/3, just relabeled.
>
> **Baseline, recomputed honestly:** my Week-4 rule is re-applied to `test_df` only (never seen during any training), then scored against `is_ctr_anomaly` with the same Precision@K function — so baseline and models are compared in the same notebook run, on the same rows, with the same metric, per the skill's non-negotiable rule.

In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

FEATURES = ["impressions_90d", "avg_position", "content_age_days", "word_count", "engagement_rate"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# --- Prepare X/y for train and test ---
X_train = train_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["is_ctr_anomaly"].values
X_test = test_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["is_ctr_anomaly"].values

# --- Model 1: Logistic Regression ---
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

# --- Model 2: Random Forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Baseline: Week-4 rule, recomputed on test_df only ---
def baseline_score(row):
    if row["avg_position"] <= 10 and row["impressions_90d"] >= 500 and row["ctr"] < 1.5:
        return row["impressions_90d"] * (1.5 - row["ctr"])
    elif 10 < row["avg_position"] <= 20 and row["impressions_90d"] >= 250 and row["ctr"] < 1.0:
        return (row["impressions_90d"] * 0.5) * (1.0 - row["ctr"])
    return 0.0

baseline_scores_test = test_df.apply(baseline_score, axis=1).values

# --- Random-chance baseline for Precision@K ---
# (the label's positive rate — what you'd expect ranking pages in random order,
# NOT majority-class accuracy, which is the wrong comparison for a ranking metric)
random_chance = y_test.mean()

# --- The comparison table ---
results = []
for k in (20, 50):
    results.append({
        "method": "Random chance (positive rate)",
        "k": k,
        "precision_at_k": round(random_chance, 3),
    })
    results.append({
        "method": "Week-4 baseline rule",
        "k": k,
        "precision_at_k": round(precision_at_k(baseline_scores_test, y_test, k), 3),
    })
    results.append({
        "method": "Logistic Regression",
        "k": k,
        "precision_at_k": round(precision_at_k(logreg_scores, y_test, k), 3),
    })
    results.append({
        "method": "Random Forest",
        "k": k,
        "precision_at_k": round(precision_at_k(rf_scores, y_test, k), 3),
    })

results_df = pd.DataFrame(results).pivot(index="method", columns="k", values="precision_at_k")
results_df.columns = [f"Precision@{k}" for k in results_df.columns]
print("Test set size:", len(test_df), "| positive rate:", round(y_test.mean(), 3))
display(results_df)

Test set size: 4610 | positive rate: 0.286


,Precision@20,Precision@50
method,,
Logistic Regression,0.550,0.480
Random Forest,0.850,0.640
Random chance (positive rate),0.286,0.286
Week-4 baseline rule,0.350,0.260


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# --- What the Random Forest leans on ---
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances.round(3).to_string())

# Sanity check: is anything suspiciously dominant (possible leakage) vs a believable spread?
top_feature_share = importances.iloc[0]
print(f"\nTop feature ('{importances.index[0]}') carries {top_feature_share:.1%} of total importance.")

# --- Concrete error examples: model's top-20 picks that are actually NOT anomalies ---
test_df_scored = test_df.copy()
test_df_scored["rf_score"] = rf_scores
test_df_scored["true_label"] = y_test

top20_model = test_df_scored.sort_values("rf_score", ascending=False).head(20)
false_positives = top20_model[top20_model["true_label"] == 0]

print(f"\n{len(false_positives)} of the model's top-20 picks are false positives (not real anomalies):")
display(false_positives[["content_id", "position_tier", "content_type", "impressions_90d",
                          "avg_position", "ctr", "rf_score"]].head(5))

Random Forest feature importances:
impressions_90d     0.372
engagement_rate     0.252
avg_position        0.236
word_count          0.080
content_age_days    0.060

Top feature ('impressions_90d') carries 37.2% of total importance.

3 of the model's top-20 picks are false positives (not real anomalies):


,content_id,position_tier,content_type,impressions_90d,avg_position,ctr,rf_score
17140,content_c8353bb0e3ef,page_3_5,feedly article,146,47.6,0.00,0.893303
18719,content_b6bb24742416,page_3_5,keyword article,227,44.0,0.44,0.892283
5444,content_d1dd4dd758df,page_3_5,keyword article,180,27.6,0.56,0.890776


> **Reading the errors:**
>
> The Random Forest's feature importances are led by `impressions_90d` (37%), `engagement_rate` (25%), and `avg_position` (24%), with `word_count` (8%) and `content_age_days` (6%) trailing. This is a believable spread — no single feature dominates, which argues against hidden leakage (a suspiciously perfect single-feature split, like `trend_pct` in Week 3's leakage trap, would be the red flag to worry about).
>
> Against a corrected random-chance baseline (the label's own positive rate, 28.6% — the right no-skill comparison for a ranking metric, not majority-class accuracy), the Week-4 hand rule performs close to chance at Precision@50 (0.260 vs 0.286) and only modestly above chance at Precision@20 (0.350 vs 0.286). Both learned models clearly separate from chance and from the rule: Logistic Regression reaches 0.550/0.480 at @20/@50, and Random Forest reaches 0.850/0.640 — roughly 2.4x the rule's precision at the top of the queue and 2.5x at 50 deep.
>
> Looking at concrete errors: all 3 of the Random Forest's top-20 false positives (`content_c8353bb0e3ef`, `content_b6bb24742416`, `content_d1dd4dd758df`) share two traits — they sit in the `page_3_5` tier, and their `impressions_90d` are near the low end of what survives the visibility filter (146–227, versus a dataset median in the thousands). This suggests the model is partly leaning on raw impression volume as a proxy for "worth flagging," and at low absolute volumes CTR estimates are noisier — a page can look anomalous just from small-sample variance. The peer-median rule underlying my label construction correctly judged these three as not-anomalous, which the model missed.
>
> **Directional takeaway:** the learned model is a real improvement over the hand-written rule for this ranking task, but it would benefit from an explicit minimum-impression floor (beyond the 100 already applied) before its top ranks are treated as high-confidence, since its errors cluster exactly where impression volume is thin.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.